# Volume Verification
Confirm Volume has been configured and docs can be ingested. 

---

In [0]:
%run ../../notebooks/_resources/Utilities

In [0]:
dbutils.widgets.text("catalog", "dennis_schultz", "Catalog Name")
dbutils.widgets.text("schema", "vantage_workshop", "Schema Name")
dbutils.widgets.text("volume", "test_documents", "Volume Name")
dbutils.widgets.text("volume_folder", "large_pdfs", "Volume Folder")
dbutils.widgets.text("bronze_table", "01_bronze", "Bronze Table Name")

# Unity Catalog
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
volume_folder = dbutils.widgets.get("volume_folder")
bronze_table = dbutils.widgets.get("bronze_table")

SOURCE_PATH = f"/Volumes/{catalog}/{schema}/{volume}/{volume_folder}"

In [0]:
# Set the default catalog and schema for this session
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

## Unstructured

## Step 1 — Load raw PDFs from the volume

In [0]:
from pyspark.sql.functions import lit

# Read all PDF files from volume as binary files

pdf_df = (spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.pdf")
    .load(SOURCE_PATH)
    .select("*", "_metadata")
    .withColumn("source", lit("Volume"))
)

display(pdf_df.limit(1))

## Step 3 — Write bronze table (raw binary + page count)

Persisting the bronze layer means downstream parsing can re-run without re-reading the volume or recomputing page counts. Bad files (`page_count = -1`) stay in bronze for visibility and are filtered before parsing.

In [0]:
bronze = (
    pdf_df
    .withColumn("file_name", element_at(split(col("path"), "/"), -1))
    .select(
        "file_name", 
        "source", 
        "path", 
        "length", 
        "modificationTime", 
        "content")
)

(
bronze.write
    .mode("append")
    .saveAsTable(bronze_table)
)

display(spark.read.table(bronze_table).drop("content"))

In [0]:
display(
    spark.read.table(bronze_table)
        .selectExpr(
            "file_name",
            "source",
            "path",
            "length",
            "modificationTime",
            "substring(content, 1, 100) as content_first_100_bytes")
)